In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.common.by import By

from time import sleep

import os



from selenium.webdriver.chrome.service import Service as ChromeService

from webdriver_manager.chrome import ChromeDriverManager



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

print("Running SE FI Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename= 'HK IAHK SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])

regulatorName = 'HK IAHK'
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

# scriptfolder=os.path.dirname(os.path.abspath(__file__))

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

#driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chromeOptions )

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {'HK IAHK 1' : 'https://www.ia.org.hk/en/supervision/reg_insurers_lloyd/register_of_authorized_insurers.html#top',
           'HK IAHK 2' : 'https://www.ia.org.hk/en/supervision/reg_ins_intermediaries/registers_of_insurance_intermediaries.html',
           'HK IAHK 3' : 'https://www.ia.org.hk/en/supervision/reg_ins_intermediaries/registers_of_insurance_intermediaries.html'}

Typology = {

            "HK IAHK 1": "Register of Authorized Insurers",

            "HK IAHK 2": "List of Licensed Insurance Agencies", 

		    "HK IAHK 3": "List of Licensed Insurance Broker Companies", 

            }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}



ISO_HK={"Hong Kong" :"HK","Bermuda" :"BM","Federal Republic of Germany" :"DE","United States of America" :"US",

        "Singapore" :"SG","China" :"CN","Italy" :"IT","Norway" :"NO","Spain" :"ES","Luxembourg" :"LU",

        "United Kingdom" :"UK","Canada" :"CA","France" :"FR","Belgium" :"BE","Isle of Man" :"IM","India" :"IN",

        "South Africa" :"ZA","Republic of Ireland" :"IE","Philippines" :"PH" ,"Japan" :"JP" ,"Guernsey" :"GG",

        "Switzerland" :"CH"}



processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def check_dowload_files(tempfolder, fileType ):



    for time in range(10):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download {fileType} file. Run Script again' )



# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    driver.get(regdict[reg])

    sleep(3)

    

    driver.find_element(By.LINK_TEXT, 'Download Full List').click()

    check_dowload_files(tempfolder, 'xlsx')

    

    filePath = os.path.join(tempfolder, os.listdir(tempfolder)[0])

    tempdf = pd.read_excel(filePath)

    os.remove(filePath)

    

    sqldict['Name'].extend(tempdf['Name of Insurer'])

    sqldict['ListProcessDate'].extend([processdate]*len(tempdf['Name of Insurer']))

    sqldict['ListName'].extend(['Insurers']*len(tempdf['Name of Insurer']))

    sqldict['ListCode'].extend(['1']*len(tempdf['Name of Insurer']))

    sqldict['RegCode'].extend(['IAHK']*len(tempdf['Name of Insurer']))

    sqldict['RegCtry'].extend(['HK']*len(tempdf['Name of Insurer']))

    sqldict['RegulationType'].extend(['Regulated']*len(tempdf['Name of Insurer']))

    sqldict['Address_1'].extend([item if item != '---' else '' for item in tempdf['Main Business Address in Hong Kong']])

    sqldict['Cntry'].extend([ISO_HK.get(item) for item in tempdf['Place of Incorporation']])

    sqldict['Phone'].extend([str(item).replace('\n','/') if str(item) != '---' else '' for item in tempdf['Tel. No.']])

    sqldict['Fax'].extend([str(item).replace('\n','/') if str(item) != '---' else '' for item in tempdf['Fax. No.']])

    sqldict['Website'].extend([str(item).replace('\n','/') if str(item) != '---' else '' for item in tempdf['Official Website Address']])

    sqldict['Email'].extend([str(item).replace('\n','/') if str(item) != '---' else '' for item in tempdf['Email Address']])

    

    # Fill the rest with empty string

    for key in sqldict.keys():

        if len(sqldict['Name']) > len(sqldict[key]):

            sqldict[key].extend(['']*len(sqldict['Name']))



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)
    
    

Running SE FI Web Scraping Tool v.1.1
[INFO] : Working 1/1 _(HK IAHK 1)_ 
[INFO] : Download xlsx file ... (wait 0/20 s)
[INFO] : xlsx file = ['register_of_insurers_as_at_07032025.xlsx'])


C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_22384\55939184.py:230: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [3]:
df.to_csv('list_1.csv')